# Adaptive NeSy-Gen: Adaptive Claim-Level Neuro-Symbolic Verification for Chest X-Ray Report Generation

**AAAI 2026 — Reproducible Experiments Notebook**

This notebook implements the complete **Adaptive NeSy-Gen** pipeline as described in the paper:

| Stage | Description |
|-------|-------------|
| 1 | Visual evidence retrieval (MedSigLIP) |
| 2 | Report drafting (MedGemma zero-shot / few-shot) |
| 3 | Clinical claim extraction & PrimeKG entity linking |
| 4 | Claim-level evidence contract |
| 5 | Adaptive routing (fast path vs. graph escalation) |
| 6 | PrimeKG radiology subgraph construction |
| 7 | LTN-style fuzzy verification (bio / diag / loc) |
| 8 | Consistency Gate (ACCEPT / REVISE / FLAG / ABSTAIN) |
| 9 | Evidence-bound selective revision |
| 10 | Faithful explanation traces |

**Important caveats:**
- MedGemma and MedSigLIP pretraining includes MIMIC-CXR. MIMIC-CXR experiments are described as *no task-specific fine-tuning*, not strict unseen-data zero-shot.
- The framework does **not** claim to eliminate hallucinations. Entity unsupported-content rates and clinical-label disagreement are used as proxies, validated with official CheXbert/RadGraph metrics.
- Explanation traces are *procedural*: they record evidence and decisions consumed during inference — not post-hoc LLM rationalizations.

## Table of Contents

1. [Environment Setup](#setup)
2. [Experiment Configuration](#config)
3. [Authentication (Hugging Face + Google Drive)](#auth)
4. [Clone Repository & Install](#install)
5. [Dataset Setup (IU X-Ray / MIMIC-CXR)](#data)
6. [PrimeKG Radiology Subgraph Cache](#primekg)
7. [Stage 1 — Visual Retrieval (MedSigLIP)](#stage1)
8. [Stage 2 — Report Drafting (MedGemma)](#stage2)
9. [Stage 3-10 — Adaptive NeSy Verification (Full Pipeline)](#stages3-10)
10. [Evaluation: Lexical Quality (BLEU / ROUGE / METEOR / CIDEr)](#eval-lexical)
11. [Evaluation: Clinical Quality (CheXbert / RadGraph)](#eval-clinical)
12. [Evaluation: Explainability & Routing Metrics](#eval-explain)
13. [Integrity Audit (Leakage / Uniqueness)](#integrity)
14. [Ablation Studies](#ablations)
15. [Visualisation: Faithful Explanation HTML Report](#visualise)
16. [Summary Table](#summary)

---
<a name="setup"></a>
## 1. Environment Setup

In [ ]:
import subprocess, sys

# Verify GPU
try:
    import torch
    if torch.cuda.is_available():
        print(f'GPU: {torch.cuda.get_device_name(0)}')
        print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    else:
        print('WARNING: No GPU detected. Switch to GPU runtime: Runtime > Change runtime type > T4 GPU')
except ImportError:
    print('torch not yet installed — will install in Cell 4')

# Python version check
print(f'Python: {sys.version}')
print('Colab runtime ready.')

---
<a name="config"></a>
## 2. Experiment Configuration

Set all experiment switches here. All downstream cells read from these variables.

In [ ]:
from pathlib import Path

# ── Dataset ──────────────────────────────────────────────────────────────────
# 'iuxray_official' : Indiana University Chest X-ray (open access, no auth)
# 'mimic_aug'       : MIMIC-CXR (requires PhysioNet credentialing)
RUN_DATASET = 'iuxray_official'

# ── Drafting backend ─────────────────────────────────────────────────────────
# 'zero_shot'  : MedGemma sees only query image + clinical indication
# 'few_shot'   : MedGemma additionally receives top-k visually retrieved reports
DRAFT_MODE = 'few_shot'

# ── Scale ────────────────────────────────────────────────────────────────────
# smoke=True runs 30 test cases to verify the pipeline end-to-end.
# Set SMOKE_RUN=False for full evaluation.
SMOKE_RUN = True
MAX_TEST_EXAMPLES   = 30    if SMOKE_RUN else None
MAX_TRAIN_INDEX     = 2000  if SMOKE_RUN else None  # MedSigLIP training index size

# ── Models ───────────────────────────────────────────────────────────────────
MEDGEMMA_MODEL  = 'google/medgemma-4b-it'
MEDSIGLIP_MODEL = 'google/medsiglip-448'

# ── Retrieval ────────────────────────────────────────────────────────────────
RETRIEVAL_TOP_K     = 5
FEW_SHOT_K          = 3    # top-k reports provided to MedGemma in few-shot mode
RETRIEVAL_BATCH_SZ  = 16

# ── Generation ───────────────────────────────────────────────────────────────
MAX_NEW_TOKENS = 180

# ── Adaptive routing thresholds (Section 5 of paper) ─────────────────────────
FAST_ACCEPT_THRESHOLD  = 0.85   # τ_fast
MIN_SUPPORTING_REPORTS = 2      # m
CLAIM_REVISE_THRESHOLD = 0.50   # τ_revise
REVISION_POLICY        = 'evidence_replace'  # or 'audit_only'

# ── Paths ────────────────────────────────────────────────────────────────────
REPO_DIR       = Path('/content/Adaptive-NeSy-Gen')
DRIVE_ROOT     = Path('/content/drive/MyDrive/adaptive_nesy_gen_aaai2026')
OUTPUT_DIR     = DRIVE_ROOT / RUN_DATASET / f'medgemma_{DRAFT_MODE}'
PRIMEKG_DIR    = DRIVE_ROOT / f'primekg_radiology_{RUN_DATASET}'
SIGLIP_CACHE   = DRIVE_ROOT / f'medsiglip_cache_{RUN_DATASET}' / 'train_index.npz'
MANIFEST_PATH  = DRIVE_ROOT / RUN_DATASET / 'manifest.jsonl'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Derived names ────────────────────────────────────────────────────────────
PRED_CSV        = OUTPUT_DIR / f'{RUN_DATASET}_{DRAFT_MODE}_predictions.csv'
CAND_CSV        = OUTPUT_DIR / f'{RUN_DATASET}_{DRAFT_MODE}_retrieval_audit.csv'
TRACE_JSONL     = OUTPUT_DIR / f'{RUN_DATASET}_{DRAFT_MODE}_claim_traces.jsonl'
CLAIM_AUDIT_CSV = OUTPUT_DIR / f'{RUN_DATASET}_{DRAFT_MODE}_claim_audit.csv'
METRICS_JSON    = OUTPUT_DIR / f'{RUN_DATASET}_{DRAFT_MODE}_metrics.json'
LEAKAGE_JSON    = OUTPUT_DIR / f'{RUN_DATASET}_{DRAFT_MODE}_leakage.json'
EXPLAIN_JSON    = OUTPUT_DIR / f'{RUN_DATASET}_{DRAFT_MODE}_explainability.json'
EXPLAIN_CSV     = OUTPUT_DIR / f'{RUN_DATASET}_{DRAFT_MODE}_explainability_claims.csv'
REPORT_HTML     = OUTPUT_DIR / f'{RUN_DATASET}_{DRAFT_MODE}_explanations.html'
ABLATION_DIR    = OUTPUT_DIR / 'ablations'
ABLATION_DIR.mkdir(parents=True, exist_ok=True)

print(f'Dataset : {RUN_DATASET}')
print(f'Mode    : MedGemma {DRAFT_MODE}')
print(f'Scale   : {"smoke (" + str(MAX_TEST_EXAMPLES) + " test)" if SMOKE_RUN else "full"}')
print(f'Output  : {OUTPUT_DIR}')
print(f'τ_fast  : {FAST_ACCEPT_THRESHOLD}  m={MIN_SUPPORTING_REPORTS}  τ_revise={CLAIM_REVISE_THRESHOLD}')

---
<a name="auth"></a>
## 3. Authentication

**Before running:**
1. Accept MedGemma & MedSigLIP terms at https://huggingface.co/google/medgemma-4b-it and https://huggingface.co/google/medsiglip-448
2. Add your `HF_TOKEN` under Colab → Secrets (key icon in left panel)

In [ ]:
import os, subprocess, sys

# Mount Google Drive — safe to re-run: skips if already mounted
from google.colab import drive
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
else:
    print('Google Drive already mounted.')

# Hugging Face authentication
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub'], check=True)
from google.colab import userdata
from huggingface_hub import login
HF_TOKEN = userdata.get('HF_TOKEN')
assert HF_TOKEN, 'Add HF_TOKEN under Colab Secrets first (🔑 icon in left panel).'
login(token=HF_TOKEN, add_to_git_credential=False)
print('Hugging Face authentication OK.')

---
<a name="install"></a>
## 4. Clone Repository & Install

In [ ]:
import subprocess, sys
from pathlib import Path

REPO_URL = 'https://github.com/FaezehMillerAI/Adaptive-NeSy-Gen.git'

if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{REPO_DIR}[torch,eval,colab]'],
    check=True
)
# Additional dependencies
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'accelerate>=0.30', 'pycocoevalcap', 'rouge-score', 'nltk'],
    check=True
)
print(f'Repository ready: {REPO_DIR}')

---
<a name="data"></a>
## 5. Dataset Setup

### IU X-Ray (default — open access)
Downloaded automatically via Kaggle. No additional authentication needed.

### MIMIC-CXR
Requires PhysioNet credentialing. Set `RUN_DATASET='mimic_aug'` and ensure your `KAGGLE_TOKEN` secret is configured.

In [ ]:
import subprocess, sys, json
from pathlib import Path

MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)

if not MANIFEST_PATH.exists():
    print(f'Building manifest for {RUN_DATASET}...')
    cmd = [
        sys.executable, 'scripts/build_manifest.py',
        '--dataset', RUN_DATASET,
        '--output', str(MANIFEST_PATH),
    ]
    subprocess.run(cmd, cwd=REPO_DIR, check=True)
else:
    print(f'Reusing existing manifest: {MANIFEST_PATH}')

# Show split statistics
with MANIFEST_PATH.open() as f:
    examples = [json.loads(line) for line in f if line.strip()]
from collections import Counter
splits = Counter(ex['split'] for ex in examples)
has_image = sum(1 for ex in examples if ex.get('image_path'))
print(f'Total examples : {len(examples)}')
print(f'Splits         : {dict(splits)}')
print(f'With image     : {has_image}')
print(f'Manifest path  : {MANIFEST_PATH}')

---
<a name="primekg"></a>
## 6. PrimeKG Radiology Subgraph Cache

PrimeKG is a large biomedical knowledge graph (≈4M edges). We build a **radiology-focused cache** once using only training-split seed entities. This cache is stored on Drive and reused for all downstream verification calls.

The cache construction is deterministic and uses only:
- Radiology-relevant node types from PrimeKG (disease, anatomy, drug, phenotype)
- Seed entities derived exclusively from the **training split** manifest
- One-hop PrimeKG neighbours of each seed entity

In [ ]:
import subprocess, sys

PRIMEKG_NODE_CSV = PRIMEKG_DIR / 'nodes.csv'
PRIMEKG_KG_CSV   = PRIMEKG_DIR / 'kg.csv'

if PRIMEKG_NODE_CSV.exists() and PRIMEKG_KG_CSV.exists():
    print(f'Reusing PrimeKG radiology cache: {PRIMEKG_DIR}')
else:
    print('Building PrimeKG radiology subgraph cache (one-time, ~10 min on first run)...')
    PRIMEKG_DIR.mkdir(parents=True, exist_ok=True)
    cmd = [
        sys.executable, 'scripts/build_radiology_primekg.py',
        '--manifest', str(MANIFEST_PATH),
        '--output-dir', str(PRIMEKG_DIR),
        '--split', 'train',
    ]
    subprocess.run(cmd, cwd=REPO_DIR, check=True)

# Inspect cache statistics
import pandas as pd
nodes = pd.read_csv(PRIMEKG_NODE_CSV)
kg    = pd.read_csv(PRIMEKG_KG_CSV, low_memory=False)
print(f'\nPrimeKG radiology cache:')
print(f'  Nodes : {len(nodes):,}  (types: {nodes["node_type"].nunique() if "node_type" in nodes.columns else "?"})')
print(f'  Edges : {len(kg):,}')
if 'node_type' in nodes.columns:
    print(nodes['node_type'].value_counts().head(10).to_string())

---
<a name="stage1"></a>
## 7. Stage 1 — Visual Evidence Retrieval (MedSigLIP)

**Formal specification:**
```
v_x = f_img(x)          # query image embedding
v_i = f_img(x_i)        # pre-cached training embeddings
s_i = cos(v_x, v_i)     # cosine similarity
```

Top-k studies are selected after excluding:
- The same underlying study (`study_id` match)
- Alternate views of that study (same `r2gen_id`)

Only training-split reports are eligible. Retrieved reports constitute **visual RAG evidence, not ground-truth supervision**.

In [ ]:
# This cell demonstrates the MedSigLIP retriever directly (for inspection).
# The full pipeline (Stages 1-10) runs in Section 9.

import sys
sys.path.insert(0, str(REPO_DIR))

import json
import numpy as np
from pathlib import Path

from nesy_gen.data.schema import load_jsonl
from nesy_gen.baselines.medsiglip_retrieval import MedSiglipRetriever

examples = load_jsonl(MANIFEST_PATH)
train_ex = [ex for ex in examples if ex.split == 'train' and ex.image_path]
test_ex  = [ex for ex in examples if ex.split == 'test'  and ex.image_path]

if MAX_TRAIN_INDEX:
    train_ex = train_ex[:MAX_TRAIN_INDEX]
if MAX_TEST_EXAMPLES:
    test_ex = test_ex[:MAX_TEST_EXAMPLES]

print(f'Training index size : {len(train_ex)}')
print(f'Query set size      : {len(test_ex)}')

# Build or reload the MedSigLIP training index
print('\nLoading frozen MedSigLIP encoder...')
retriever = MedSiglipRetriever(MEDSIGLIP_MODEL)

# Run retrieval (builds and caches the training index on first call)
print('Running visual retrieval...')
retrieved_lists = retriever.retrieve(
    train_ex, test_ex,
    top_k=RETRIEVAL_TOP_K,
    batch_size=RETRIEVAL_BATCH_SZ,
    cache_path=SIGLIP_CACHE,
)

# Show retrieval statistics
import pandas as pd
rows = [
    {'query_id': test_ex[i].study_id, 'rank': r.rank,
     'retrieved_id': r.retrieved_study_id, 'similarity': r.similarity}
    for i, neighbours in enumerate(retrieved_lists)
    for r in neighbours
]
ret_df = pd.DataFrame(rows)
print(f'\nRetrieval summary (top-1 similarities):')
rank1 = ret_df[ret_df['rank'] == 1]
print(rank1['similarity'].describe().round(3))
print(f'\nExample top-3 neighbours for query {test_ex[0].study_id}:')
print(ret_df[ret_df['query_id'] == test_ex[0].study_id].to_string(index=False))

---
<a name="stage2"></a>
## 8. Stage 2 — Report Drafting (MedGemma)

**Zero-shot mode:**
```
y_raw = G(x, indication)
```

**Few-shot (retrieval-conditioned) mode:**
```
y_raw = G(x, indication, R_k(x))
```
where `R_k(x)` contains reports from the top-k visually similar training radiographs.

The prompt explicitly:
- Restricts output to the Findings section
- Requires preservation of negation and laterality
- Prohibits copying findings unsupported by the current image
- Treats retrieved reports as non-authoritative style/evidence examples

In [ ]:
from nesy_gen.models.medgemma import MedGemmaDrafter, build_medgemma_prompt

# Show the prompt template for transparency
example_indication = 'Evaluate for pneumonia. Patient with fever and productive cough.'
example_evidence   = [
    'Bilateral lower lobe infiltrates consistent with pneumonia.',
    'No pleural effusion. Heart size normal.',
]

print('=== Zero-shot prompt ===')
print(build_medgemma_prompt(example_indication))
print()
print('=== Few-shot prompt (top-2 evidence reports) ===')
print(build_medgemma_prompt(example_indication, example_evidence))

print('\n--- Loading MedGemma (this downloads ~9 GB on first run) ---')
import torch
torch.cuda.empty_cache()
drafter = MedGemmaDrafter(MEDGEMMA_MODEL)
print('MedGemma loaded.')

# Draft a report for the first test example
q = test_ex[0]
evidence_reports = (
    [r.prediction for r in retrieved_lists[0][:FEW_SHOT_K]]
    if DRAFT_MODE == 'few_shot' else []
)
draft = drafter.draft(
    q.image_path,
    indication=q.indication,
    evidence_reports=evidence_reports,
    max_new_tokens=MAX_NEW_TOKENS,
)
print(f'\nQuery study  : {q.study_id}')
print(f'Indication   : {q.indication or "(none)"}')
print(f'Draft ({DRAFT_MODE}): {draft}')
print(f'Reference    : {q.report[:200]}...' if q.report else 'Reference    : (unavailable)')

---
<a name="stages3-10"></a>
## 9. Stages 3–10 — Full Adaptive NeSy-Gen Pipeline

This cell runs all remaining stages on the complete test set:

| Stage | Key output |
|-------|------------|
| 3 | Claims extracted; entities linked to PrimeKG | 
| 4 | Evidence record `E(c_j) = [s_visual, s_retrieval, n_support, s_KG, s_LTN, s_gate]` |
| 5 | Fast-path vs. escalation routing decision |
| 6 | Compact PrimeKG radiology subgraph per uncertain claim |
| 7 | Fuzzy LTN clause scores: τ_bio, τ_diag, τ_loc → τ(c_j) |
| 8 | Gate decision ∈ {ACCEPT, REVISE, FLAG, ABSTAIN} |
| 9 | Selective evidence-bound revision (entity-matched, polarity-preserving) |
| 10 | Per-claim `ClaimEvidenceTrace` saved to JSONL |

**Conservative revision policy:** a replacement is only permitted when the retrieved sentence has identical PrimeKG entity IDs, identical assertion polarity, introduces no new entities, and exceeds τ_revise. Otherwise the original claim is preserved and marked FLAGGED.

In [ ]:
import os, json, time, subprocess, sys
import pandas as pd
from tqdm.auto import tqdm

from nesy_gen.agents.adaptive_verification import AdaptiveClaimVerifier, split_clinical_claims
from nesy_gen.generation.rag import RagCandidate
from nesy_gen.models.pipeline_factory import build_primekg_pipeline

print('Loading adaptive PrimeKG/LTN verification pipeline...')
pipeline = build_primekg_pipeline(str(PRIMEKG_DIR))
verifier = AdaptiveClaimVerifier(
    pipeline,
    fast_accept_threshold=FAST_ACCEPT_THRESHOLD,
    min_supporting_reports=MIN_SUPPORTING_REPORTS,
    revise_threshold=CLAIM_REVISE_THRESHOLD,
    revision_policy=REVISION_POLICY,
)
print('Pipeline ready.')

# Run end-to-end
selected_rows, candidate_rows, trace_rows, claim_rows = [], [], [], []
t0 = time.perf_counter()

for example, neighbours in tqdm(
    zip(test_ex, retrieved_lists, strict=True),
    total=len(test_ex), desc='Adaptive NeSy-Gen'
):
    rag_candidates = [
        RagCandidate(
            source='medsiglip_retrieval',
            source_rank=row.rank,
            prediction=row.prediction,
            evidence_score=max(0.0, min(1.0, row.similarity)),
            retrieved_study_id=row.retrieved_study_id,
        )
        for row in neighbours
    ]
    evidence_reports = (
        [c.prediction for c in rag_candidates[:FEW_SHOT_K]]
        if DRAFT_MODE == 'few_shot' else []
    )

    # Stage 2: draft
    draft = drafter.draft(
        example.image_path,
        indication=example.indication,
        evidence_reports=evidence_reports,
        max_new_tokens=MAX_NEW_TOKENS,
    )

    # Stages 3-10: adaptive claim verification
    visual_support = max((c.evidence_score for c in rag_candidates), default=0.0)
    result = verifier.verify(
        draft,
        indication=example.indication,
        visual_support=visual_support,
        evidence_candidates=rag_candidates,
    )

    selected_rows.append({
        'study_id': example.study_id,
        'prediction': result.final_report,
        'raw_prediction': draft,
        'reference': example.report,
        'source': f'medgemma_{DRAFT_MODE}',
        'accepted_claims': result.accepted_claims,
        'revised_claims': result.revised_claims,
        'flagged_claims': result.flagged_claims,
        'graph_calls': result.graph_calls,
        'total_claims': result.total_claims,
        'escalation_rate': result.escalation_rate,
        'adaptive_latency_ms': result.latency_ms,
    })
    candidate_rows.extend({
        'study_id': example.study_id,
        'source': c.source, 'rank': c.source_rank,
        'retrieved_id': c.retrieved_study_id,
        'evidence_score': c.evidence_score,
    } for c in rag_candidates)
    trace_rows.append({'study_id': example.study_id, **result.as_dict()})
    claim_rows.extend({'study_id': example.study_id, **cl.as_dict()} for cl in result.claims)

elapsed = time.perf_counter() - t0
print(f'\nCompleted {len(selected_rows)} reports in {elapsed:.1f}s ({elapsed/len(selected_rows):.2f}s/report)')

# Save outputs
pred_df  = pd.DataFrame(selected_rows)
cand_df  = pd.DataFrame(candidate_rows)
claim_df = pd.DataFrame(claim_rows)
pred_df.to_csv(PRED_CSV,  index=False)
cand_df.to_csv(CAND_CSV,  index=False)
claim_df.to_csv(CLAIM_AUDIT_CSV, index=False)
with TRACE_JSONL.open('w') as f:
    for row in trace_rows:
        f.write(json.dumps(row) + '\n')
print(f'Predictions  : {PRED_CSV}')
print(f'Claim traces : {TRACE_JSONL}')

# Quick pipeline summary
print(f'\n=== Pipeline summary ===')
print(pred_df[['accepted_claims','revised_claims','flagged_claims',
               'total_claims','escalation_rate']].mean().round(3))

---
<a name="eval-lexical"></a>
## 10. Evaluation — Lexical Quality

Metrics: **BLEU-1/2/3/4**, **ROUGE-L**, **METEOR**, **CIDEr**

These measure surface-level text overlap against reference reports. They are necessary but not sufficient for clinical quality assessment.

In [ ]:
import json
import subprocess, sys

result = subprocess.run(
    [
        sys.executable, 'scripts/evaluate_generation.py',
        '--manifest',         str(MANIFEST_PATH),
        '--predictions-csv',  str(PRED_CSV),
        '--output-json',      str(METRICS_JSON),
        '--nodes-csv',        str(PRIMEKG_DIR / 'nodes.csv'),
        '--output-factuality-csv', str(OUTPUT_DIR / 'entity_factuality.csv'),
        '--output-chexpert-csv',   str(OUTPUT_DIR / 'chexpert_quick.csv'),
        '--output-radgraph-csv',   str(OUTPUT_DIR / 'radgraph_quick.csv'),
    ],
    cwd=REPO_DIR, capture_output=True, text=True
)
if result.returncode != 0:
    print('STDERR:', result.stderr[-2000:])
    raise RuntimeError('evaluate_generation.py failed')

metrics = json.loads(METRICS_JSON.read_text())
lex = metrics.get('lexical_metrics', {})

print('=== Lexical Quality Metrics ===')
for k, v in lex.items():
    print(f'  {k:<12} : {v:.4f}')

# Show also raw vs. verified comparison
import pandas as pd
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer

pred_df = pd.read_csv(PRED_CSV)
valid = pred_df[pred_df['reference'].notna() & pred_df['reference'].str.strip().ne('')]

scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
smooth = SmoothingFunction().method1

raw_rouges, ver_rouges = [], []
for _, row in valid.iterrows():
    ref = str(row['reference']).lower().split()
    raw = str(row['raw_prediction']).lower().split()
    ver = str(row['prediction']).lower().split()
    raw_rouges.append(scorer.score(' '.join(ref), ' '.join(raw))['rougeL'].fmeasure)
    ver_rouges.append(scorer.score(' '.join(ref), ' '.join(ver))['rougeL'].fmeasure)

print(f'\nROUGE-L raw draft  : {sum(raw_rouges)/len(raw_rouges):.4f}')
print(f'ROUGE-L verified   : {sum(ver_rouges)/len(ver_rouges):.4f}')
delta = (sum(ver_rouges) - sum(raw_rouges)) / len(raw_rouges)
print(f'Delta              : {delta:+.4f}')

---
<a name="eval-clinical"></a>
## 11. Evaluation — Clinical Quality

### 11a. Entity-Level Clinical Metrics
Precision / recall / F1 over PrimeKG-linked clinical entities.

### 11b. Official CheXbert / RadGraph
The official CheXbert and RadGraph tools require separate installation. This section prepares the input files and shows the quick internal proxy metrics. For final paper results, run the official tools on the prepared input files.

In [ ]:
import pandas as pd
import json

metrics = json.loads(METRICS_JSON.read_text())

# Entity factuality proxy metrics
fact = metrics.get('factuality_metrics', {})
print('=== Entity Factuality Proxy Metrics ===')
for k, v in fact.items():
    if isinstance(v, float):
        print(f'  {k:<30} : {v:.4f}')

# Negation consistency
neg = metrics.get('negation_metrics', {})
if neg:
    print('\n=== Negation Consistency ===')
    for k, v in neg.items():
        print(f'  {k:<30} : {v}')

# Prepare official inputs
print('\n--- Preparing official CheXbert/RadGraph input files ---')
OFFICIAL_DIR = OUTPUT_DIR / 'official_metric_inputs'
subprocess.run(
    [
        sys.executable, 'scripts/evaluate_generation.py',
        '--manifest',         str(MANIFEST_PATH),
        '--predictions-csv',  str(PRED_CSV),
        '--output-json',      str(OUTPUT_DIR / 'official_input_prep.json'),
        '--nodes-csv',        str(PRIMEKG_DIR / 'nodes.csv'),
        '--prepare-official-inputs-dir', str(OFFICIAL_DIR),
    ],
    cwd=REPO_DIR, check=True
)
print(f'Official inputs ready at: {OFFICIAL_DIR}')
print('Files:', list(OFFICIAL_DIR.glob('*'))[:10])

print("""
=== Next step for official metrics ===
1. CheXbert F1:
   python -m chexpert_labeler --reports_path {}/reports.csv
   Then compare predictions vs. references with label overlap F1.

2. RadGraph F1:
   python radgraph/inference.py --reports_path {}/reports.csv
   Use the official RadGraph F1 scorer.
""".format(OFFICIAL_DIR, OFFICIAL_DIR))

---
<a name="eval-explain"></a>
## 12. Evaluation — Explainability & Routing Metrics

Per the paper's explainability evaluation:

| Metric | Description |
|--------|-------------|
| Linked-claim coverage | Fraction of claims with ≥1 PrimeKG entity |
| Entity-specific grounding | Fraction with entity-level retrieval evidence |
| Adaptive escalation rate | Fraction of linked claims routed to graph verification |
| Graph-path coverage | Fraction of escalated claims with PrimeKG path |
| Gate-decision distribution | Counts of ACCEPT/REVISE/FLAG/ABSTAIN |
| Revision rate | Fraction of claims revised by selective intervention |
| Mean claim latency | Per-claim verification time (ms) |

In [ ]:
import subprocess, sys, json
import pandas as pd
import matplotlib.pyplot as plt

subprocess.run(
    [
        sys.executable, 'scripts/evaluate_adaptive_explanations.py',
        '--trace-jsonl', str(TRACE_JSONL),
        '--output-json', str(EXPLAIN_JSON),
        '--output-csv',  str(EXPLAIN_CSV),
    ],
    cwd=REPO_DIR, check=True
)

explain = json.loads(EXPLAIN_JSON.read_text())
print('=== Explainability Metrics ===')
for k, v in explain.items():
    if k != 'decision_counts':
        print(f'  {k:<40} : {v}')

# Gate decision distribution
decisions = explain.get('decision_counts', {})
if decisions:
    print('\n=== Consistency Gate Decision Distribution ===')
    total = sum(decisions.values())
    for label, count in sorted(decisions.items(), key=lambda x: -x[1]):
        bar = '█' * int(30 * count / total)
        print(f'  {label:<30} {count:5d}  {count/total:.1%}  {bar}')

# Plot claim-level latency distribution
claim_df = pd.read_csv(CLAIM_AUDIT_CSV)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(claim_df['latency_ms'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Claim latency (ms)')
axes[0].set_ylabel('Count')
axes[0].set_title('Per-claim verification latency')

decision_counts = claim_df['decision'].value_counts()
colors = {'accept_fast_path': '#2ecc71', 'accept_verified': '#27ae60',
          'revise': '#f39c12', 'flag': '#e74c3c', 'abstain': '#95a5a6'}
bar_colors = [colors.get(k, '#3498db') for k in decision_counts.index]
axes[1].bar(decision_counts.index, decision_counts.values, color=bar_colors)
axes[1].set_xlabel('Gate Decision')
axes[1].set_ylabel('Count')
axes[1].set_title('Consistency Gate decision distribution')
plt.xticks(rotation=30, ha='right')

plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'explainability_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Plot saved: {OUTPUT_DIR}/explainability_plots.png')

---
<a name="integrity"></a>
## 13. Integrity Audit

Verifies that predictions are not memorised training reports:

| Check | Description |
|-------|-------------|
| Exact-match leakage | Predictions identical to a training report |
| High-overlap audit | ROUGE-L ≥ 0.95 with any training report |
| Same-study exclusion | Retrieved neighbours never from same study |
| Unique-prediction ratio | Fraction of predictions that are unique |
| Diversity (type-token ratio) | Lexical diversity of generated text |

In [ ]:
import subprocess, sys, json

subprocess.run(
    [
        sys.executable, 'scripts/audit_prediction_leakage.py',
        '--manifest',              str(MANIFEST_PATH),
        '--predictions-csv',       str(PRED_CSV),
        '--output-json',           str(LEAKAGE_JSON),
        '--high-overlap-threshold', '0.95',
    ],
    cwd=REPO_DIR, check=True
)

leakage = json.loads(LEAKAGE_JSON.read_text())
print('=== Prediction Integrity Audit ===')
for k, v in leakage.items():
    print(f'  {k:<40} : {v}')

# Additional diversity check
import pandas as pd
pred_df = pd.read_csv(PRED_CSV)
unique_ratio = pred_df['prediction'].nunique() / len(pred_df)
all_tokens = ' '.join(pred_df['prediction'].fillna('').tolist()).split()
ttr = len(set(all_tokens)) / len(all_tokens) if all_tokens else 0.0
avg_len = pred_df['prediction'].fillna('').apply(lambda x: len(x.split())).mean()
print(f'\n  unique_prediction_ratio        : {unique_ratio:.4f}')
print(f'  type_token_ratio               : {ttr:.4f}')
print(f'  mean_prediction_length_words   : {avg_len:.1f}')

---
<a name="ablations"></a>
## 14. Ablation Studies

The ablation suite evaluates all configurations described in Section 13 of the paper:

| # | Configuration |
|---|---------------|
| 1 | Retrieval-only baseline (top-1 retrieved report) |
| 2 | MedGemma zero-shot (no retrieval, no verification) |
| 3 | MedGemma few-shot (retrieval, no verification) |
| 4 | RAG without PrimeKG/LTN (retrieval + gate, no graph) |
| 5 | Report-level PrimeKG/LTN (not claim-level) |
| 6 | Adaptive claim audit without revision (audit_only) |
| 7 | **Full Adaptive NeSy-Gen** (claim verification + revision) |
| 8 | Always-on verification (no fast path, τ_fast = 0) |
| 9 | Adaptive without LTN (PrimeKG connectivity only) |
| 10 | Adaptive without Consistency Gate |
| 11 | Shuffled PrimeKG relations (control) |
| 12 | τ_fast = 0.70 (lower threshold) |
| 13 | τ_fast = 0.95 (higher threshold) |

In [ ]:
import subprocess, sys, json
import pandas as pd

ABLATION_CONFIGS = [
    # (name, extra_args)
    ('retrieval_only',         ['--draft-mode', 'zero_shot', '--limit', '1',
                                '--ablation', 'retrieval_only']),
    ('medgemma_zero_shot',     ['--draft-mode', 'zero_shot', '--ablation', 'no_verify']),
    ('medgemma_few_shot_rag',  ['--draft-mode', 'few_shot',  '--ablation', 'no_verify']),
    ('rag_no_graph',           ['--draft-mode', 'few_shot',  '--ablation', 'no_graph']),
    ('report_level_verify',    ['--draft-mode', 'few_shot',  '--ablation', 'report_level']),
    ('audit_only',             ['--draft-mode', 'few_shot',  '--revision-policy', 'audit_only']),
    ('full_adaptive',          ['--draft-mode', 'few_shot']),   # default = full system
    ('always_on_verify',       ['--draft-mode', 'few_shot',  '--fast-accept-threshold', '0.0']),
    ('no_ltn',                 ['--draft-mode', 'few_shot',  '--ablation', 'no_ltn']),
    ('no_gate',                ['--draft-mode', 'few_shot',  '--ablation', 'no_gate']),
    ('shuffled_primekg',       ['--draft-mode', 'few_shot',  '--ablation', 'shuffled_kg']),
    ('tau_fast_0.70',          ['--draft-mode', 'few_shot',  '--fast-accept-threshold', '0.70']),
    ('tau_fast_0.95',          ['--draft-mode', 'few_shot',  '--fast-accept-threshold', '0.95']),
]

ablation_results = {}

for name, extra_args in ABLATION_CONFIGS:
    abl_out = ABLATION_DIR / name
    abl_out.mkdir(exist_ok=True)
    abl_pred = abl_out / 'predictions.csv'
    abl_trace = abl_out / 'claim_traces.jsonl'
    abl_metrics = abl_out / 'metrics.json'

    if abl_metrics.exists():
        print(f'[SKIP] {name} — metrics already computed')
    else:
        print(f'[RUN ] {name}')
        cmd = [
            sys.executable, 'scripts/run_ablation_suite.py',
            '--manifest',          str(MANIFEST_PATH),
            '--primekg-dir',       str(PRIMEKG_DIR),
            '--output-dir',        str(abl_out),
            '--retrieval-cache',   str(SIGLIP_CACHE),
            '--medgemma-model',    MEDGEMMA_MODEL,
            '--medsiglip-model',   MEDSIGLIP_MODEL,
            '--nodes-csv',         str(PRIMEKG_DIR / 'nodes.csv'),
        ]
        if MAX_TEST_EXAMPLES:
            cmd += ['--limit', str(MAX_TEST_EXAMPLES)]
        cmd += extra_args
        r = subprocess.run(cmd, cwd=REPO_DIR, capture_output=True, text=True)
        if r.returncode != 0:
            print(f'  WARNING: {name} failed: {r.stderr[-300:]}')
            continue

    if abl_metrics.exists():
        m = json.loads(abl_metrics.read_text())
        ablation_results[name] = {
            **m.get('lexical_metrics', {}),
            **{f'explain_{k}': v for k, v in m.get('explainability', {}).items()
               if isinstance(v, (int, float))},
        }

# Pretty ablation table
abl_df = pd.DataFrame(ablation_results).T
display_cols = ['bleu_4', 'rouge_l', 'meteor', 'cider']
display_cols = [c for c in display_cols if c in abl_df.columns]
if display_cols:
    print('\n=== Ablation Results ===')
    print(abl_df[display_cols].round(4).to_string())

abl_df.to_csv(ABLATION_DIR / 'ablation_summary.csv')
print(f'\nAblation summary saved: {ABLATION_DIR}/ablation_summary.csv')

---
<a name="visualise"></a>
## 15. Visualisation: Faithful Explanation HTML Report

Renders per-claim explanation traces as an interactive HTML report. Each claim shows:
- Original and final text
- Linked PrimeKG entities with node IDs and types
- Report-level visual similarity and entity-specific retrieval grounding
- Whether graph verification was triggered
- LTN clause scores (τ_bio, τ_diag, τ_loc) and aggregate τ(c_j)
- PrimeKG graph path used for verification
- Consistency Gate decision with reason
- Replacement provenance (if revised)

In [ ]:
import subprocess, sys
from IPython.display import HTML, display

subprocess.run(
    [
        sys.executable, 'scripts/build_adaptive_explanation_report.py',
        '--trace-jsonl',  str(TRACE_JSONL),
        '--output-html',  str(REPORT_HTML),
        '--limit',        '20',
    ],
    cwd=REPO_DIR, check=True
)

print(f'HTML report: {REPORT_HTML}')
display(HTML(REPORT_HTML.read_text()))

---
## 15b. Inline Explanation Viewer (single study)

Interactive per-claim breakdown for a single study.

In [ ]:
import json, pandas as pd
from IPython.display import HTML, display

# Load traces
traces = []
with TRACE_JSONL.open() as f:
    for line in f:
        traces.append(json.loads(line))

# Select a trace with verification triggered
interesting = [
    t for t in traces
    if any(c.get('verification_triggered') for c in t.get('claims', []))
]
trace = interesting[0] if interesting else traces[0]
print(f'Showing study: {trace["study_id"]}')
print(f'Original: {trace["original_report"]}')
print(f'Verified: {trace["final_report"]}')

# Build inline HTML
decision_colors = {
    'accept_fast_path': '#d5f5e3',
    'accept_verified':  '#a9dfbf',
    'revise':           '#fdebd0',
    'flag':             '#fadbd8',
    'abstain':          '#eaecee',
}

rows_html = ''
for c in trace.get('claims', []):
    color = decision_colors.get(c.get('decision', ''), '#ffffff')
    entities = ', '.join(
        f"{e['node_name']} ({e['node_type']})"
        for e in c.get('linked_entities', [])
    ) or '—'
    ltn = c.get('ltn_truth')
    ltn_str = f'{ltn:.3f}' if ltn is not None else 'skipped (fast path)'
    triggered = '✓' if c.get('verification_triggered') else '—'
    rows_html += f"""
    <tr style="background:{color}">
      <td>{c['claim_id']}</td>
      <td>{c['original_claim']}</td>
      <td>{c['final_claim']}</td>
      <td>{entities}</td>
      <td>{c.get('visual_support', 0):.3f}</td>
      <td>{c.get('retrieval_support', 0):.3f}</td>
      <td>{c.get('retrieval_support_count', 0)}</td>
      <td>{triggered}</td>
      <td>{ltn_str}</td>
      <td>{c.get('primekg_status', '—')}</td>
      <td><b>{c.get('decision', '—')}</b></td>
      <td>{c.get('reason', '—')}</td>
      <td>{c.get('latency_ms', 0):.1f} ms</td>
    </tr>"""

html = f"""
<style>table{{border-collapse:collapse;font-size:12px}}th,td{{border:1px solid #ccc;padding:4px 8px}}</style>
<h3>Study: {trace['study_id']}</h3>
<table>
<tr><th>#</th><th>Original claim</th><th>Final claim</th><th>PrimeKG entities</th>
<th>s_visual</th><th>s_retrieval</th><th>n_support</th><th>Graph?</th>
<th>τ(c)</th><th>PrimeKG status</th><th>Decision</th><th>Reason</th><th>Latency</th></tr>
{rows_html}
</table>"""
display(HTML(html))

---
<a name="summary"></a>
## 16. Summary Table

Aggregates all results into the paper's main results table.

In [ ]:
import json, pandas as pd

metrics  = json.loads(METRICS_JSON.read_text())
explain  = json.loads(EXPLAIN_JSON.read_text())
leakage  = json.loads(LEAKAGE_JSON.read_text())
pred_df  = pd.read_csv(PRED_CSV)
claim_df = pd.read_csv(CLAIM_AUDIT_CSV)

lex = metrics.get('lexical_metrics', {})

summary = {
    # ── Lexical ──────────────────────────────────────────────────────────────
    'BLEU-1':  lex.get('bleu_1', float('nan')),
    'BLEU-2':  lex.get('bleu_2', float('nan')),
    'BLEU-3':  lex.get('bleu_3', float('nan')),
    'BLEU-4':  lex.get('bleu_4', float('nan')),
    'ROUGE-L': lex.get('rouge_l', float('nan')),
    'METEOR':  lex.get('meteor', float('nan')),
    'CIDEr':   lex.get('cider', float('nan')),
    # ── Routing ──────────────────────────────────────────────────────────────
    'Escalation rate (all claims)':    pred_df['escalation_rate'].mean(),
    'Accept (fast path)':    (claim_df['decision'] == 'accept_fast_path').mean(),
    'Accept (verified)':     (claim_df['decision'] == 'accept_verified').mean(),
    'Revised':               (claim_df['decision'] == 'revise').mean(),
    'Flagged/Abstained':     claim_df['decision'].isin(['flag', 'abstain']).mean(),
    # ── Explainability ───────────────────────────────────────────────────────
    'Linked-claim coverage':        explain.get('linked_claim_rate', float('nan')),
    'Entity grounding coverage':    explain.get('grounded_claim_rate', float('nan')),
    'Graph-path coverage (escal)':  explain.get('graph_path_coverage_when_escalated', float('nan')),
    'Mean claim latency (ms)':      explain.get('mean_claim_latency_ms', float('nan')),
    # ── Integrity ────────────────────────────────────────────────────────────
    'Exact-match leakage':          leakage.get('exact_match_count', 0),
    'High-overlap (≥0.95) count':   leakage.get('high_overlap_count', 0),
    # ── Efficiency ───────────────────────────────────────────────────────────
    'Mean graph calls / report':    pred_df['graph_calls'].mean(),
    'Mean total claims / report':   pred_df['total_claims'].mean(),
    'Mean adaptive latency (ms)':   pred_df['adaptive_latency_ms'].mean(),
}

summary_df = pd.DataFrame.from_dict(
    summary, orient='index', columns=[f'MedGemma {DRAFT_MODE} ({RUN_DATASET})']
)
print('\n===  Adaptive NeSy-Gen — Full Results Summary  ===')
print(summary_df.round(4).to_string())
summary_df.to_csv(OUTPUT_DIR / 'full_results_summary.csv')
print(f'\nSaved: {OUTPUT_DIR}/full_results_summary.csv')

---
## 17. Computational Efficiency Report

Measures all efficiency metrics described in the paper.

In [ ]:
import torch, time, os
import pandas as pd

pred_df  = pd.read_csv(PRED_CSV)
claim_df = pd.read_csv(CLAIM_AUDIT_CSV)

print('=== Computational Efficiency ===')

# GPU memory
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1e9
    reserved  = torch.cuda.memory_reserved()  / 1e9
    total     = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'  GPU memory allocated : {allocated:.2f} GB')
    print(f'  GPU memory reserved  : {reserved:.2f} GB')
    print(f'  GPU memory total     : {total:.2f} GB')

# Index size
if SIGLIP_CACHE.exists():
    cache_mb = SIGLIP_CACHE.stat().st_size / 1e6
    print(f'  MedSigLIP index size : {cache_mb:.1f} MB')

# Latency statistics
print(f'\n  Mean claim latency (ms)   : {claim_df["latency_ms"].mean():.2f}')
print(f'  P50 claim latency (ms)    : {claim_df["latency_ms"].quantile(0.50):.2f}')
print(f'  P95 claim latency (ms)    : {claim_df["latency_ms"].quantile(0.95):.2f}')
print(f'  Mean adaptive latency (ms): {pred_df["adaptive_latency_ms"].mean():.2f}')
print(f'  Mean graph calls/report   : {pred_df["graph_calls"].mean():.2f}')
print(f'  Mean total claims/report  : {pred_df["total_claims"].mean():.2f}')

fast = claim_df[claim_df['decision'] == 'accept_fast_path']['latency_ms']
graph = claim_df[claim_df['verification_triggered'].astype(bool)]['latency_ms']
if len(fast) > 0:
    print(f'\n  Fast-path mean latency (ms) : {fast.mean():.2f} (n={len(fast)})')
if len(graph) > 0:
    print(f'  Graph-path mean latency (ms): {graph.mean():.2f} (n={len(graph)})')

---
## 18. Assumptions, Limitations & Claim Boundaries

This cell documents the explicit assumptions and limitations described in the paper.

**Entity linker:** The lexical linker is deterministic and reproducible. Its accuracy must be evaluated separately using mention precision, linking accuracy, negation accuracy, coverage, and manual error analysis. Entity linking errors propagate to all downstream stages.

**LTN scores:** The aggregate τ(c_j) measures satisfaction of the implemented graph constraints. It is **not** a direct probability that the clinical statement is true.

**Explanation traces:** These are procedural explanations recording evidence and decisions consumed during inference. They are **not** post-hoc natural-language rationalizations and should **not** be described as complete causal explanations of the neural generator.

**Hallucination:** The system does not claim to eliminate hallucinations. Entity unsupported-content rates and clinical-label disagreement are used as proxies. Any hallucination-reduction claim requires official clinical metric outputs, appropriate statistical testing, and preferably expert review.

**MIMIC-CXR:** MedGemma and MedSigLIP pretraining likely includes MIMIC-CXR. MIMIC experiments are *no task-specific fine-tuning*, not strict unseen-data zero-shot.

**PrimeKG coverage:** Some radiology entities and relations are absent from PrimeKG. Unlinked claims are marked ABSTAIN and excluded from graph verification statistics. This must be reported separately.

In [ ]:
import pandas as pd

claim_df = pd.read_csv(CLAIM_AUDIT_CSV)

total_claims  = len(claim_df)
linked_claims = (claim_df['num_entities'] > 0 if 'num_entities' in claim_df.columns
                 else claim_df['linked_entities'].apply(
                     lambda x: len(eval(x)) if isinstance(x, str) else 0) > 0)

abstain       = (claim_df['decision'] == 'abstain').sum()
abstain_rate  = abstain / total_claims if total_claims else 0.0

print('=== Linker Coverage & Abstain Rate ===')
print(f'  Total claims     : {total_claims}')
print(f'  ABSTAIN (unlinked): {abstain} ({abstain_rate:.1%})')
print()
print('These unlinked claims are excluded from graph verification.')
print('Report escalation rates over:')
print(f'  (a) all claims        : {pred_df["escalation_rate"].mean():.3f}')
linked_esc = claim_df[
    claim_df['decision'] != 'abstain'
]['verification_triggered'].astype(bool).mean() if 'verification_triggered' in claim_df.columns else float('nan')
print(f'  (b) linked claims only: {linked_esc:.3f}')

---
## All outputs

All artifacts have been saved to Google Drive at:

```
MyDrive/adaptive_nesy_gen_aaai2026/{dataset}/medgemma_{mode}/
  ├── *_predictions.csv          — final and raw predictions
  ├── *_retrieval_audit.csv      — per-query retrieval evidence
  ├── *_claim_traces.jsonl       — faithful explanation traces (per-claim)
  ├── *_claim_audit.csv          — per-claim summary table
  ├── *_metrics.json             — lexical + entity metrics
  ├── *_leakage.json             — integrity audit
  ├── *_explainability.json      — explainability summary
  ├── *_explanations.html        — interactive explanation report
  ├── full_results_summary.csv   — main paper results table
  ├── explainability_plots.png   — gate distribution + latency plots
  └── ablations/                 — ablation suite outputs
      └── ablation_summary.csv
```

**Claim boundary:** The main contribution of this work is adaptive, claim-level neuro-symbolic verification with inference-faithful evidence traces and selective intervention. Performance improvements should be interpreted with reference to official clinical metrics, appropriate baselines, and the limitations documented in Section 18.